# CO$_2$ - binning

Bin the observational network.

## Imports

In [1]:
from pathlib import Path

import openscm_units
import pandas as pd
import pint
from pydoit_nb.config_handling import get_config_for_step_id

import local.binned_data_interpolation
import local.binning
import local.raw_data_processing
from local.config import load_config_from_file

In [2]:
pint.set_application_registry(openscm_units.unit_registry)  # type: ignore

## Define branch this notebook belongs to

In [3]:
step: str = "calculate_co2_monthly_fifteen_degree_pieces"

## Parameters

In [4]:
config_file: str = "../../dev-config-absolute.yaml"  # config file
step_config_id: str = "only"  # config ID to select for this branch

In [5]:
# Parameters
config_file = "/Users/znicholls/Documents/repos/CMIP-GHG-Concentration-Generation/output-bundles/v1.0.0/v1.0.0-config.yaml"
step_config_id = "only"


## Load config

In [6]:
config = load_config_from_file(Path(config_file))
config_step = get_config_for_step_id(config=config, step=step, step_config_id=step_config_id)

config_process_noaa_surface_flask_data = get_config_for_step_id(
    config=config,
    step="process_noaa_surface_flask_data",
    step_config_id=config_step.gas,
)
config_process_noaa_in_situ_data = get_config_for_step_id(
    config=config,
    step="process_noaa_in_situ_data",
    step_config_id=config_step.gas,
)

## Action

### Load data

In [7]:
all_data_l = []
for f, dep_short_names in [
    (
        config_process_noaa_surface_flask_data.processed_monthly_data_with_loc_file,
        local.dependencies.load_source_info_short_names(
            config_process_noaa_surface_flask_data.source_info_short_names_file
        ),
    ),
    (
        config_process_noaa_in_situ_data.processed_monthly_data_with_loc_file,
        local.dependencies.load_source_info_short_names(
            config_process_noaa_in_situ_data.source_info_short_names_file
        ),
    ),
]:
    try:
        all_data_l.append(local.raw_data_processing.read_and_check_binning_columns(f))
    except Exception as exc:
        msg = f"Error reading {f}"
        raise ValueError(msg) from exc

    for dsn in dep_short_names:
        local.dependencies.save_dependency_into_db(
            db=config.dependency_db,
            gas=config_step.gas,
            dependency_short_name=dsn,
        )

all_data = pd.concat(all_data_l)
all_data["gas"] = all_data["gas"].str.lower()
all_data = all_data[all_data["gas"] == config_step.gas]
all_data

,gas,reporting_id,year,month,latitude,longitude,value,unit,site_code_filename,site_code,surf_or_ship,source,network,station,measurement_method
0,co2,month,1968,1,40.050,-105.630,323.650,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask
1,co2,month,1968,2,40.050,-105.630,325.070,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask
2,co2,month,1968,3,40.050,-105.630,326.050,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask
3,co2,month,1968,4,40.050,-105.630,326.550,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask
4,co2,month,1968,5,40.050,-105.630,326.400,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2344,co2,MonthlyData,2024,1,-14.247,-170.564,420.980,ppm,smo,SMO,surface,insitu,NOAA,smo,insitu
2345,co2,MonthlyData,2024,2,-14.247,-170.564,421.740,ppm,smo,SMO,surface,insitu,NOAA,smo,insitu
2346,co2,MonthlyData,2024,3,-14.247,-170.564,421.390,ppm,smo,SMO,surface,insitu,NOAA,smo,insitu
2347,co2,MonthlyData,2024,4,-14.247,-170.564,419.970,ppm,smo,SMO,surface,insitu,NOAA,smo,insitu


## Bin and average data

- all measurements from a station are first averaged for the month
- then average over all stations
    - stations get equal weight
    - flask/in situ networks (i.e. different measurement methods/techniques)
      are treated as separate stations i.e. get equal weight
- this order is best as you have a better chance of avoiding giving different times more weight by accident
    - properly equally weighting all times in the month would be very hard,
      because you'd need to interpolate to a super fine grid first (one for future research)

In [8]:
all_data_with_bins = local.binning.add_lat_lon_bin_columns(all_data)
all_data_with_bins

,gas,reporting_id,year,month,latitude,longitude,value,unit,site_code_filename,site_code,surf_or_ship,source,network,station,measurement_method,lon_bin,lat_bin
0,co2,month,1968,1,40.050,-105.630,323.650,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask,-90.0,37.5
1,co2,month,1968,2,40.050,-105.630,325.070,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask,-90.0,37.5
2,co2,month,1968,3,40.050,-105.630,326.050,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask,-90.0,37.5
3,co2,month,1968,4,40.050,-105.630,326.550,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask,-90.0,37.5
4,co2,month,1968,5,40.050,-105.630,326.400,ppm,nwr,NWR,surface,flask,NOAA,nwr,flask,-90.0,37.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2344,co2,MonthlyData,2024,1,-14.247,-170.564,420.980,ppm,smo,SMO,surface,insitu,NOAA,smo,insitu,-150.0,-7.5
2345,co2,MonthlyData,2024,2,-14.247,-170.564,421.740,ppm,smo,SMO,surface,insitu,NOAA,smo,insitu,-150.0,-7.5
2346,co2,MonthlyData,2024,3,-14.247,-170.564,421.390,ppm,smo,SMO,surface,insitu,NOAA,smo,insitu,-150.0,-7.5
2347,co2,MonthlyData,2024,4,-14.247,-170.564,419.970,ppm,smo,SMO,surface,insitu,NOAA,smo,insitu,-150.0,-7.5


In [9]:
print(local.binning.get_network_summary(all_data_with_bins))

Collating data from:
- NOAA flask (94 stations: 000, abp, alt, ams ... uum, wis, wlg, zep)
- NOAA insitu (5 stations: brw, mko, mlo, smo, spo)


In [10]:
bin_averages = local.binning.calculate_bin_averages(all_data_with_bins)
bin_averages

Will ignore columns: ['reporting_id', 'latitude', 'longitude', 'site_code_filename', 'site_code', 'surf_or_ship', 'source']
Took mean over ['index']
Took mean over ['measurement_method', 'network', 'station']


,gas,unit,year,month,lat_bin,lon_bin,value
0,co2,ppm,1968,1,37.5,-90.0,323.65
1,co2,ppm,1968,2,37.5,-90.0,325.07
2,co2,ppm,1968,3,37.5,-90.0,326.05
3,co2,ppm,1968,4,37.5,-90.0,326.55
4,co2,ppm,1968,5,37.5,-90.0,326.40
...,...,...,...,...,...,...,...
16401,co2,ppm,2024,3,67.5,-150.0,430.27
16402,co2,ppm,2024,4,-82.5,-30.0,418.82
16403,co2,ppm,2024,4,-7.5,-150.0,419.97
16404,co2,ppm,2024,4,22.5,-150.0,426.69


### Save

In [11]:
local.binned_data_interpolation.check_data_columns_for_binned_data_interpolation(bin_averages)
assert set(bin_averages["gas"]) == {config_step.gas}

In [12]:
config_step.processed_bin_averages_file.parent.mkdir(exist_ok=True, parents=True)
bin_averages.to_csv(config_step.processed_bin_averages_file, index=False)
bin_averages

,gas,unit,year,month,lat_bin,lon_bin,value
0,co2,ppm,1968,1,37.5,-90.0,323.65
1,co2,ppm,1968,2,37.5,-90.0,325.07
2,co2,ppm,1968,3,37.5,-90.0,326.05
3,co2,ppm,1968,4,37.5,-90.0,326.55
4,co2,ppm,1968,5,37.5,-90.0,326.40
...,...,...,...,...,...,...,...
16401,co2,ppm,2024,3,67.5,-150.0,430.27
16402,co2,ppm,2024,4,-82.5,-30.0,418.82
16403,co2,ppm,2024,4,-7.5,-150.0,419.97
16404,co2,ppm,2024,4,22.5,-150.0,426.69
